# MNIST Handwritten Digit Classification using TensorFlow/Keras

**Author:** _add your name_

**Objective:** The objective of this project is to build a neural network using TensorFlow/Keras to classify handwritten digits from 0 to 9 using the MNIST dataset.

This notebook walks through the complete workflow: loading and exploring the data, training a baseline model, running one controlled experiment (**adding Dropout**), and comparing the two models on the same test set.

Every number and every figure below is produced by running the code in this notebook. Nothing is hard-coded, and the conclusion is written from the measured results rather than from an assumption that Dropout must help.

**How to run:** use *Kernel -> Restart Kernel and Run All Cells* so the whole notebook executes in order from a clean state.

---
## 1. Import the libraries

NumPy is used for arrays, Matplotlib for the figures, TensorFlow/Keras for the neural network, scikit-learn for the classification report and confusion matrix, and `pathlib`/`json` for saving the results.

In [ ]:
import json
import pathlib
import platform
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix

print('TensorFlow version:', tf.__version__)
print('NumPy version     :', np.__version__)
print('Python version    :', platform.python_version())
print('GPU available     :', bool(tf.config.list_physical_devices('GPU')))

### Reproducibility and shared settings

All random seeds are fixed, and both models use exactly the same training settings. Keeping everything except the Dropout layer identical is what makes the experiment a fair, controlled comparison.

In [ ]:
SEED = 42


def set_seeds():
    """Fix the Python, NumPy and TensorFlow random seeds."""
    random.seed(SEED)
    np.random.seed(SEED)
    tf.keras.utils.set_random_seed(SEED)


set_seeds()

# Training settings, identical for both models.
EPOCHS = 10
BATCH_SIZE = 128
VALIDATION_SPLIT = 0.1   # 10% of the training data is held out for validation
HIDDEN_UNITS = 128
DROPOUT_RATE = 0.5

# Output folders.
PROJECT_ROOT = pathlib.Path.cwd()
RESULTS_DIR = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

print('Figures and metrics will be saved to:', RESULTS_DIR)
print('Model will be saved to              :', MODELS_DIR)

---
## 2. Load the MNIST dataset

Keras downloads MNIST automatically the first time and caches it locally.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print('Training images:', x_train.shape)
print('Training labels:', y_train.shape)
print('Test images    :', x_test.shape)
print('Test labels    :', y_test.shape)

---
## 3. Explore the dataset

Before modelling it helps to know how much data there is, what the images look like, what range the pixel values cover and how balanced the classes are.

In [ ]:
print('Number of training samples:', x_train.shape[0])
print('Number of test samples    :', x_test.shape[0])
print('Image dimensions          :', x_train.shape[1:], '(height x width, grayscale)')
print('Number of classes         :', len(np.unique(y_train)))
print('Pixel value range         :', x_train.min(), 'to', x_train.max())
print('Unique labels             :', np.unique(y_train))
print()
print('Images per digit in the training set:')
for digit in range(10):
    print(f'  digit {digit}: {(y_train == digit).sum()}')

---
## 4. Sample MNIST images

One example of each of the ten digit classes, so we can see what the network has to learn from.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
indices = [int(np.where(y_train == digit)[0][0]) for digit in range(10)]

for ax, index in zip(axes.flat, indices):
    ax.imshow(x_train[index], cmap='gray')
    ax.set_title(f'Label: {y_train[index]}')
    ax.axis('off')

fig.suptitle('Sample MNIST images (one per class)', fontsize=14)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'sample_images.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Preprocess the data

The raw pixel values are integers between 0 and 255. Dividing by 255 rescales them to the 0.0-1.0 range, which makes training faster and more stable. The labels stay as integers (0-9), which is what `sparse_categorical_crossentropy` expects.

In [ ]:
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print('New pixel value range:', x_train.min(), 'to', x_train.max())
print('Data type            :', x_train.dtype)
print('First ten labels     :', y_train[:10])

---
## 6. Build the baseline model

A small, deliberately simple fully-connected network:

```text
Input 28x28 -> Flatten (784) -> Dense 128 + ReLU -> Dense 10 + Softmax
```

- **Flatten** turns each 28x28 image into a single vector of 784 numbers.
- **Dense(128, ReLU)** is the hidden layer that learns useful features.
- **Dense(10, Softmax)** outputs a probability for each digit class.

In [ ]:
def build_baseline_model():
    """A small fully-connected classifier with no regularisation."""
    model = keras.Sequential(
        [
            layers.Input(shape=(28, 28), name='image'),
            layers.Flatten(name='flatten'),
            layers.Dense(HIDDEN_UNITS, activation='relu', name='hidden'),
            layers.Dense(10, activation='softmax', name='output'),
        ],
        name='baseline_mlp',
    )
    return model


set_seeds()  # reset the seed so the comparison starts from a known state
baseline_model = build_baseline_model()
baseline_model.summary()

---
## 7. Compile and train the baseline model

We use the Adam optimiser and sparse categorical cross-entropy. 10% of the training data is held out as a validation set so we can watch for overfitting; the test set is left completely untouched until the final evaluation.

In [ ]:
baseline_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

start = time.perf_counter()
baseline_history = baseline_model.fit(
    x_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=2,
)
baseline_train_time = time.perf_counter() - start

print(f'\nBaseline training time: {baseline_train_time:.1f} seconds')

---
## 8. Training history

Training accuracy, validation accuracy, training loss and validation loss for the baseline model.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
epochs_range = range(1, EPOCHS + 1)

axes[0].plot(epochs_range, baseline_history.history['accuracy'], marker='o', label='Training accuracy')
axes[0].plot(epochs_range, baseline_history.history['val_accuracy'], marker='s', label='Validation accuracy')
axes[0].set_title('Baseline accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, baseline_history.history['loss'], marker='o', label='Training loss')
axes[1].plot(epochs_range, baseline_history.history['val_loss'], marker='s', label='Validation loss')
axes[1].set_title('Baseline loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(RESULTS_DIR / 'training_history.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Evaluate the baseline on the test set

The test set has not been seen during training. We report the loss and accuracy, then look at the confusion matrix and the per-class report to see which digits the model finds difficult.

In [ ]:
baseline_test_loss, baseline_test_accuracy = baseline_model.evaluate(x_test, y_test, verbose=0)
print(f'Baseline test loss    : {baseline_test_loss:.4f}')
print(f'Baseline test accuracy: {baseline_test_accuracy * 100:.2f}%')

baseline_probabilities = baseline_model.predict(x_test, verbose=0)
baseline_predictions = np.argmax(baseline_probabilities, axis=1)
cm = confusion_matrix(y_test, baseline_predictions)

fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(cm, cmap='Blues')
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
ax.set_title('Confusion matrix (baseline, test set)')
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_xticks(range(10))
ax.set_yticks(range(10))
for row in range(10):
    for column in range(10):
        ax.text(
            column,
            row,
            cm[row, column],
            ha='center',
            va='center',
            fontsize=8,
            color='white' if cm[row, column] > cm.max() / 2 else 'black',
        )
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print(classification_report(y_test, baseline_predictions, digits=4))

---
## 10. Predictions on five test images

Each title shows the true label, the predicted label and the model's confidence, and is coloured green when the prediction is correct and red when it is wrong.

In [ ]:
predictions = baseline_model.predict(x_test[:5], verbose=0)
predicted_classes = np.argmax(predictions, axis=1)

fig, axes = plt.subplots(1, 5, figsize=(15, 4))
for i, ax in enumerate(axes):
    ax.imshow(x_test[i], cmap='gray')
    confidence = predictions[i][predicted_classes[i]] * 100
    correct = predicted_classes[i] == y_test[i]
    ax.set_title(
        f'True: {y_test[i]}\n'
        f'Pred: {predicted_classes[i]} ({confidence:.1f}%)\n'
        f"{'correct' if correct else 'wrong'}",
        color='green' if correct else 'red',
    )
    ax.axis('off')

fig.suptitle('Predictions on five test images', fontsize=14)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'predictions.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 11. Experiment: adding Dropout

**Hypothesis:** adding a `Dropout(0.5)` layer will reduce overfitting and improve test accuracy.

The architecture, optimiser, loss, epochs, batch size and validation split are all identical to the baseline - the Dropout layer is the *only* change. Dropout randomly switches off half of the hidden units during each training step, which discourages the network from relying on any single unit.

The seed is reset before building this model so that both networks start from the same initial weights.

In [ ]:
def build_dropout_model():
    """The identical network with a Dropout layer after the hidden layer."""
    model = keras.Sequential(
        [
            layers.Input(shape=(28, 28), name='image'),
            layers.Flatten(name='flatten'),
            layers.Dense(HIDDEN_UNITS, activation='relu', name='hidden'),
            layers.Dropout(DROPOUT_RATE, name='dropout'),
            layers.Dense(10, activation='softmax', name='output'),
        ],
        name='dropout_mlp',
    )
    return model


set_seeds()  # same starting weights as the baseline, for a fair comparison
dropout_model = build_dropout_model()
dropout_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
dropout_model.summary()

start = time.perf_counter()
dropout_history = dropout_model.fit(
    x_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VALIDATION_SPLIT,
    verbose=2,
)
dropout_train_time = time.perf_counter() - start

print(f'\nDropout model training time: {dropout_train_time:.1f} seconds')

---
## 12. Compare the baseline and the Dropout model

Both models are evaluated on the same untouched test set.

In [ ]:
dropout_test_loss, dropout_test_accuracy = dropout_model.evaluate(x_test, y_test, verbose=0)

print(f'Baseline test accuracy: {baseline_test_accuracy * 100:.2f}%  (loss {baseline_test_loss:.4f})')
print(f'Dropout  test accuracy: {dropout_test_accuracy * 100:.2f}%  (loss {dropout_test_loss:.4f})')
print(f'Difference (Dropout - Baseline): {(dropout_test_accuracy - baseline_test_accuracy) * 100:+.2f} percentage points')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs_range, baseline_history.history['val_accuracy'], marker='o', label='Baseline')
axes[0].plot(epochs_range, dropout_history.history['val_accuracy'], marker='s', label='With Dropout')
axes[0].set_title('Validation accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, baseline_history.history['val_loss'], marker='o', label='Baseline')
axes[1].plot(epochs_range, dropout_history.history['val_loss'], marker='s', label='With Dropout')
axes[1].set_title('Validation loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

accuracies = [baseline_test_accuracy * 100, dropout_test_accuracy * 100]
bars = axes[2].bar(['Baseline', 'With Dropout'], accuracies, color=['#4C72B0', '#DD8452'], width=0.55)
axes[2].set_title('Test accuracy')
axes[2].set_ylabel('Accuracy (%)')
axes[2].set_ylim(min(accuracies) - 2, 100)
for bar, accuracy in zip(bars, accuracies):
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                 f'{accuracy:.2f}%', ha='center', va='bottom')
axes[2].grid(alpha=0.3, axis='y')

fig.suptitle('Experiment: baseline vs adding Dropout', fontsize=14)
fig.tight_layout()
fig.savefig(RESULTS_DIR / 'experiment_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 13. Save the best model

Whichever model achieved the higher test accuracy is saved as `models/mnist_model.keras`.

In [ ]:
if dropout_test_accuracy >= baseline_test_accuracy:
    best_model, best_model_name = dropout_model, 'Model with Dropout'
else:
    best_model, best_model_name = baseline_model, 'Baseline model'

model_path = MODELS_DIR / 'mnist_model.keras'
best_model.save(model_path)

print('Saved model       :', model_path)
print('Best model        :', best_model_name)
print(f'Its test accuracy : {max(baseline_test_accuracy, dropout_test_accuracy) * 100:.2f}%')

---
## 14. Save the metrics and fill in the report

The measured values are written to `results/metrics.json`, and the same numbers are used to fill the placeholders in `report/MNIST_Report.md`. This is why the written report cannot contain a number that the code did not produce.

In [ ]:
metrics = {
    'python_version': platform.python_version(),
    'tensorflow_version': tf.__version__,
    'seed': SEED,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'validation_split': VALIDATION_SPLIT,
    'dropout_rate': DROPOUT_RATE,
    'hidden_units': HIDDEN_UNITS,
    'num_train': int(x_train.shape[0]),
    'num_test': int(x_test.shape[0]),
    'num_classes': int(len(np.unique(y_train))),
    'baseline': {
        'parameters': int(baseline_model.count_params()),
        'test_loss': float(baseline_test_loss),
        'test_accuracy': float(baseline_test_accuracy),
        'train_time_seconds': float(baseline_train_time),
        'final_val_accuracy': float(baseline_history.history['val_accuracy'][-1]),
        'final_val_loss': float(baseline_history.history['val_loss'][-1]),
    },
    'dropout': {
        'parameters': int(dropout_model.count_params()),
        'test_loss': float(dropout_test_loss),
        'test_accuracy': float(dropout_test_accuracy),
        'train_time_seconds': float(dropout_train_time),
        'final_val_accuracy': float(dropout_history.history['val_accuracy'][-1]),
        'final_val_loss': float(dropout_history.history['val_loss'][-1]),
    },
    'best_model': best_model_name,
    'accuracy_difference_points': float((dropout_test_accuracy - baseline_test_accuracy) * 100),
}

(RESULTS_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
(RESULTS_DIR / 'classification_report.txt').write_text(
    classification_report(y_test, baseline_predictions, digits=4), encoding='utf-8'
)
print('Saved results/metrics.json and results/classification_report.txt')

# Reuse the report writer from the training script so both entry points agree.
import sys

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from train_model import write_report

report_path = write_report(metrics)
print('Report written to:', report_path)

---
## 15. Conclusion

The verdict below is derived from the numbers this notebook actually measured. There is nothing to edit here - the run decides the wording.

In [ ]:
difference = (dropout_test_accuracy - baseline_test_accuracy) * 100

if abs(difference) < 0.1:
    verdict = (
        'The two models performed essentially the same on the test set '
        f'(a difference of only {difference:+.2f} percentage points, which is within the '
        'run-to-run noise of a single training run). In this setting Dropout therefore did '
        'not meaningfully improve accuracy.'
    )
elif difference > 0:
    verdict = (
        'Adding Dropout improved test accuracy by '
        f'{difference:.2f} percentage points compared with the baseline, so in this '
        'setting it was a genuine improvement.'
    )
else:
    verdict = (
        'Adding Dropout reduced test accuracy by '
        f'{abs(difference):.2f} percentage points compared with the baseline. The '
        'experiment therefore does not support the claim that Dropout improves performance '
        'on this task with these settings.'
    )

print(f'Baseline test accuracy : {baseline_test_accuracy * 100:.2f}%')
print(f'Dropout  test accuracy : {dropout_test_accuracy * 100:.2f}%')
print()
print('CONCLUSION:', verdict)
print()
print('A single run is not enough to prove one architecture is reliably better;')
print('several runs with different seeds would be needed for a strong claim.')

---
## 16. Reproducing this project

```bash
git clone <repository-url>
cd mnist-digit-classification
python -m venv .venv
# Windows: .venv\Scripts\activate    |    macOS/Linux: source .venv/bin/activate
pip install -r requirements.txt
```

Then either open this notebook and choose *Restart Kernel and Run All*, or run the equivalent script:

```bash
python src/train_model.py
```

Both paths produce the same outputs:

| File | Contents |
| --- | --- |
| `results/sample_images.png` | One example of each digit class |
| `results/training_history.png` | Baseline training vs validation accuracy and loss |
| `results/predictions.png` | Predictions on five test images |
| `results/experiment_comparison.png` | Baseline vs Dropout comparison |
| `results/confusion_matrix.png` | Confusion matrix on the test set |
| `results/metrics.json` | All measured metrics |
| `models/mnist_model.keras` | The best trained model |
| `report/MNIST_Report.md` | The written report, filled with the measured numbers |

---
## 17. Final explanation and conclusion

**What the project does.** This notebook builds, trains and evaluates a small feed-forward
neural network with TensorFlow/Keras that classifies handwritten digits (0-9) from the
MNIST dataset, and then runs one controlled experiment that adds a Dropout layer to see
whether it improves test accuracy.

**How MNIST works.** MNIST contains 70,000 grayscale images of handwritten digits, each
28x28 pixels with an integer label 0-9. The standard split is 60,000 images for training
and 10,000 for testing, which is exactly how the dataset is loaded in Section 2.

**Preprocessing.** The only preprocessing needed is scaling: raw pixels are integers in
the range 0-255, and dividing by 255 turns them into floats in the range 0.0-1.0, which
makes training faster and more stable (Section 5). The labels stay as integers 0-9
because the model uses sparse categorical cross-entropy. The images stay 28x28;
flattening to a 784-value vector happens inside the model with a Flatten layer.

**Neural network architecture.** The baseline model is
Input 28x28 -> Flatten (784) -> Dense 128 (ReLU) -> Dense 10 (Softmax): a simple,
beginner-friendly fully-connected network with just over 100,000 parameters (see the
model summary in Section 6). The experiment model is identical except that a
Dropout(0.5) layer is inserted before the output layer (Section 11).

**Training process.** Both models are trained for 10 epochs with batch size 128, using
the Adam optimiser, sparse categorical cross-entropy as the loss and accuracy as the
metric. 10% of the training data is held out as a validation set, and the training
history is stored so the curves in Section 8 and Section 12 can be plotted. The seed is
reset before each model so both start from the same initial weights, which makes the
comparison fair.

**Test accuracy.** The measured test loss and test accuracy for both models are printed
in Section 9 and Section 12. The exact numbers are deliberately not repeated in this
text, so the written conclusion can never drift away from what the code actually
produced.

**What the graphs show.** The accuracy plot shows training and validation accuracy
rising epoch by epoch, and the loss plot shows the corresponding loss falling. If the
validation loss started rising while the training loss kept falling, that would be
overfitting. The comparison figure in Section 12 plots both models' validation curves
side by side together with their final test accuracy.

**How predictions were made.** For every test image the model outputs ten probabilities
(one per digit, summing to 1) through the softmax layer; the predicted digit is the one
with the highest probability (argmax). The five test images in Section 10 show the true
label, the predicted label and the model's confidence, coloured green when correct and
red when wrong. The confusion matrix in Section 9 shows, for every digit 0-9, how often
each digit was classified as every other digit.

**Conclusion.** The final verdict is computed in Section 15 from the measured difference
between the two models - whether Dropout helped, hurt, or made no meaningful difference
in this run. A single run cannot prove that one architecture is reliably better; several
runs with different seeds would be needed for a strong claim. Either way, the baseline
network reaches the high test accuracy printed in Section 9, which completes the
objective of this project.
